In [1]:
import numpy as np
import tensorflow as tf
import keras
import matplotlib.pyplot as plt

import sys

sys.path.append("../")

In [2]:
import numpy as np

seq_length = 60
dim = 2


def sample_circle(n, max_n=seq_length, dim=2):
    points = np.full((max_n, dim), -1, dtype=np.float32)
    sample_points = np.random.randn(n, dim)
    sample_points /= np.linalg.norm(sample_points, axis=1, keepdims=True)
    sample_points *= np.random.uniform(0, 1)  # Scale by random radius
    points[:n, :] = sample_points
    return points


def sample_square(n, max_n=seq_length, dim=2):
    points = np.full((max_n, dim), -1, dtype=np.float32)
    sample_points = np.random.uniform(-1, 1, (n, dim))
    sample_points /= np.linalg.norm(sample_points, axis=1, keepdims=True, ord=np.inf)
    sample_points *= np.random.uniform(0, 1)  # Scale by random radius
    points[:n, :] = sample_points
    return points


dataset_size = 15000
X_sphere = np.zeros((dataset_size, seq_length, dim), dtype=np.float32)
X_square = np.zeros((dataset_size, seq_length, dim), dtype=np.float32)

for iter in range(dataset_size):
    n = np.random.randint(1, seq_length + 1)
    X_sphere[iter, :, :] = sample_circle(n, seq_length, dim)
    X_square[iter, :, :] = sample_square(n, seq_length, dim)

X = np.concatenate([X_sphere, X_square], axis=0)
y = np.concatenate(
    [np.zeros((dataset_size, 1), dtype=np.float32), np.ones((dataset_size, 1), dtype=np.float32)],
    axis=0,
)

np.random.seed(42)
shuffle_indices = np.random.permutation(X.shape[0])
X = X[shuffle_indices]
y = y[shuffle_indices]


In [ ]:
from src.model.components import (
    MultiHeadAttentionBlock,
    MultiHeadAttentionStack,
    SelfAttentionBlock,
    SelfAttentionStack,
    PoolingAttentionBlock,
    GenerateMask,
    GetSequenceLength,
    DecoderQueries,
    GenerateDecoderMask,
    MLP,
    MaskOutput,
)

input_dim = dim
feature_dim = 6
latent_dim = 24
num_seeds = latent_dim // feature_dim
dropout_rate = 0.0
num_heads = 4

pixel_input = keras.layers.Input(shape=(seq_length, input_dim), name="pixel_input")
pixel_mask = GenerateMask(name="pixel_mask")(pixel_input)
pixel_seqlen = GetSequenceLength(name="pixel_seqlen")(pixel_mask)

pixel_embedding = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="pixel_embedding",
)(pixel_input)

pixel_self_attention = SelfAttentionStack(
    num_heads=num_heads,
    key_dim=feature_dim,
    stack_size=3,
    name="pixel_self_attention",
)(pixel_embedding, pixel_mask)

pixel_pooling = PoolingAttentionBlock(
    key_dim=feature_dim,
    num_seeds=num_seeds,
    num_heads=num_heads,
    name="pixel_pooling",
)(pixel_self_attention, pixel_mask)

pixel_latent_space = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="pixel_latent_space",
)(pixel_pooling)

encoded_seqlen = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="encoded_seqlen",
)(pixel_seqlen)
encoded_seqlen = keras.layers.RepeatVector(
    num_seeds, name="encoded_seqlen_repeat"
)(encoded_seqlen)

pixel_latent_space = keras.layers.Add(name="pixel_latent_space_add")(
    [pixel_latent_space, encoded_seqlen]
)

pixel_latent_output = keras.layers.Flatten(name="pixel_latent_output")(
    pixel_latent_space
)


decoder_queries = DecoderQueries(
    num_queries=seq_length,
    feature_dim=feature_dim,
    name="decoder_queries",
)(pixel_seqlen)

decoded_embedded_queries = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="decoded_embedded_queries",
)(decoder_queries)


decoder_mask = GenerateDecoderMask(name="decoder_mask", max_length=seq_length)(
    pixel_seqlen
)

decoded_latent_set = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="decoded_latent_set",
)(pixel_latent_space)


decoded_pixel_space = MultiHeadAttentionStack(
    num_heads=num_heads,
    key_dim=feature_dim,
    stack_size=3,
    name="decoded_pixel_space",
)(decoder_queries, decoded_latent_set, query_mask=decoder_mask)


decoded_point_set_self_attention = SelfAttentionStack(
    num_heads=num_heads,
    key_dim=feature_dim,
    stack_size=3,
    name="decoded_point_set_self_attention",
)(decoded_pixel_space, decoder_mask)

decoded_point_set = MLP(
    num_layers=4,
    output_dim=input_dim,
    name="decoded_point_set_mlp",
    activation="linear",
)(decoded_point_set_self_attention)

decoded_output = MaskOutput(name="decoded_output", padding_value=-1)(
    decoded_point_set, decoder_mask
)

"""
seqlen_output = keras.layers.Concatenate(name="seqlen_output")(
    [
        keras.layers.Flatten(name="seqlen_flatten")(pixel_seqlen),
        keras.layers.Flatten(name="decoder_seqlen_flatten")(decoder_seqlen),
    ]
)"""

SetAE = keras.Model(
    inputs=pixel_input,
    outputs=[decoded_output],
    name="SetAutoEncoder",
)

from src.training import SplitMSE, ChamferDistanceMasked, EmbeddingSpaceSpreading

SetAE.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss={"decoded_output": ChamferDistanceMasked(padding_value=-1)},
)

SetAE.summary()
SetAE.fit(
    X_square,
    X_square,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True
        )
    ],
)

Model: "SetAutoEncoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ pixel_input         │ (None, 60, 2)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_mask          │ (None, 60, 1)     │          0 │ pixel_input[0][0] │
│ (GenerateMask)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_embedding     │ (None, 60, 6)     │        122 │ pixel_input[0][0] │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_seqlen        │ (None, 1)         │          0 │ pixel_mask[0][0]  │
│ (GetSequenceLength) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_self_attenti… │ (None, 60, 6)     │      2,520 │ pixel_embedding[… │
│ (SelfAttentionStac… │                   │            │ pixel_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_pooling       │ (None, 4, 6)      │        864 │ pixel_self_atten… │
│ (PoolingAttentionB… │                   │            │ pixel_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoded_seqlen      │ (None, 6)         │         78 │ pixel_seqlen[0][… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_latent_space  │ (None, 4, 6)      │        336 │ pixel_pooling[0]… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoded_seqlen_rep… │ (None, 4, 6)      │          0 │ encoded_seqlen[0… │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_latent_space… │ (None, 4, 6)      │          0 │ pixel_latent_spa… │
│ (Add)               │                   │            │ encoded_seqlen_r… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_queries     │ (None, 60, 6)     │        360 │ pixel_seqlen[0][… │
│ (DecoderQueries)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_latent_set  │ (None, 4, 6)      │        336 │ pixel_latent_spa… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_mask        │ (None, 60, 1)     │          0 │ pixel_seqlen[0][… │
│ (GenerateDecoderMa… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_pixel_space │ (None, 60, 6)     │      2,520 │ decoder_queries[… │
│ (MultiHeadAttentio… │                   │            │ decoded_latent_s… │
│                     │                   │            │ decoder_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_point_set_… │ (None, 60, 6)     │      2,520 │ decoded_pixel_sp… │
│ (SelfAttentionStac… │                   │            │ decoder_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_point_set_… │ (None, 60, 2)     │        114 │ decoded_point_se… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 9,439 (36.87 KB)

 Trainable params: 9,439 (36.87 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


/Users/simi/mu3e_trigger/venv/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'ffn_layer' (of type Sequential) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/Users/simi/mu3e_trigger/venv/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'pixel_pooling' (of type PoolingAttentionBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/Users/simi/mu3e_trigger/venv/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'decoded_point_set_mlp' (of type MLP) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream l

375/375 ━━━━━━━━━━━━━━━━━━━━ 16s 25ms/step - loss: 0.3206 - val_loss: 0.2594
Epoch 2/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 9s 24ms/step - loss: 0.2426 - val_loss: 0.2307
Epoch 3/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - loss: 0.2133 - val_loss: 0.2015
Epoch 4/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - loss: 0.1987 - val_loss: 0.1880
Epoch 5/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - loss: 0.1853 - val_loss: 0.1821
Epoch 6/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - loss: 0.1793 - val_loss: 0.1747
Epoch 7/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - loss: 0.1770 - val_loss: 0.1740
Epoch 8/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - loss: 0.1740 - val_loss: 0.1739
Epoch 9/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 12s 31ms/step - loss: 0.1714 - val_loss: 0.1665
Epoch 10/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.1747

In [ ]:
sample = X_square[0:1]
decoded = SetAE.predict(sample)


In [ ]:
sample = X_square[0:2]
decoded = SetAE.predict(sample)
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(sample[0, :, 0], sample[0, :, 1], label="Input", color="blue")
ax.scatter(decoded[0, :, 0], decoded[0, :, 1], label="Decoded", color="red")
ax.set_title("Input vs Decoded Output")
ax.set_xlabel("X-axis")
ax.set_ylabel("Y-axis")
ax.legend()
plt.show()


In [ ]:
from src.model.components import (
    MultiHeadAttentionStack,
    SelfAttentionStack,
    PoolingAttentionBlock,
    GenerateMask,
    MLP,
)

feature_dim = 6
num_heads = 4


input = keras.layers.Input(shape=(seq_length, dim), name="input")
mask = GenerateMask(name="mask")(input)
input_embedding = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="pixel_attention",
)(input)
self_attention = SelfAttentionStack(
    num_heads=num_heads,
    key_dim=feature_dim,
    stack_size=3,
    name="self_attention",
)(input_embedding, mask)
pooling = PoolingAttentionBlock(
    key_dim=feature_dim,
    num_seeds=1,
    name="pooling",
)(self_attention, mask)
latent_space =keras.layers.Flatten(name="latent_space")(pooling)
output = MLP(
    num_layers=4,
    output_dim=1,
    activation="sigmoid",
    name="output",
)(latent_space)

model = keras.Model(
    inputs=input,
    outputs=output,
    name="ClassificationModel",
)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy()],
)

In [ ]:
model.fit(
    X,
    y,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True
        )
    ],
    shuffle=True,
)